# 01 — ChEMBL molecular pretraining dataset

**Goal:** Prepare a **structure-only**, reproducible molecular dataset for self-supervised graph neural network (GINE) pretraining. We will later fine-tune the encoder on proton-pump inhibition (pIC₅₀).

**Key safeguards:** reserve PPI molecules from pretraining, reject invalid structures, remove salts/duplicate connectivity, record software/release versions, and use a separate pretraining validation set. **This notebook does not train a GNN or calculate predictive R².**

Run in the `ppi_gnn` VS Code kernel. Start with `N_SAMPLE=5000` as a smoke test, then increase to `100000` and repeat from the sampling cell. Outputs go to `data/processed/`.

## 0. Install dependencies and obtain ChEMBL structures

In a **terminal**, inside `ppi_gnn`:

```bash
conda activate ppi_gnn
python -m pip install pandas numpy rdkit scikit-learn tqdm
mkdir -p data/raw data/processed
wget -c -O data/raw/chembl_37_chemreps.txt.gz \
  https://ftp.ebi.ac.uk/pub/databases/chembl/ChEMBLdb/releases/chembl_37/chembl_37_chemreps.txt.gz
```

This is the ChEMBL **37** chemical representations (SMILES/InChI/ID), *not* the full activity database. The file is about 293 MB compressed. The release is **pinned**, rather than using `latest`, so the experiment remains reproducible. If the file already exists, simply point `CHEMREPS` to it. ChEMBL's file is TSV with the columns `chembl_id`, `canonical_smiles`, `standard_inchi`, and `standard_inchi_key`.

Keep your existing `PPI_molecule_summary_248.csv` in the same folder as this notebook (or change `PPI_CSV`). Downloading is an explicit terminal step, not automatic notebook behavior.

In [10]:
from pathlib import Path
import gzip, json, random, hashlib, sys
from collections import Counter
import numpy as np
import pandas as pd
import rdkit
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import train_test_split

SEED = 42
N_SAMPLE = 100_000           # Smoke test first; then set to 100_000
RESERVOIR_MULTIPLIER = 1.5 # Draw extra SMILES because some fail filters
VAL_FRACTION = 0.1
PPI_CSV = Path('PPI_molecule_summary_248.csv')
CHEMREPS = Path('data/raw/chembl_37_chemreps.txt.gz')
OUT = Path('data/processed')
OUT.mkdir(parents=True, exist_ok=True)

for fp in [PPI_CSV, CHEMREPS]:
    if not fp.is_file():
        raise FileNotFoundError(f'Missing {fp.resolve()} — see setup above')

print('Python:', sys.version.split()[0], 'RDKit:', rdkit.__version__)
print('PPI dataset:', PPI_CSV.resolve())
print('ChEMBL file:', CHEMREPS.resolve(), 'size MB:', round(CHEMREPS.stat().st_size/1e6, 1))

Python: 3.11.18 RDKit: 2026.09.1
PPI dataset: /home/kjana/kalyanashis/ML/alfatrainig/deep_learning_bootcamp/project/GNN/PPI_molecule_summary_248.csv
ChEMBL file: /home/kjana/kalyanashis/ML/alfatrainig/deep_learning_bootcamp/project/GNN/data/raw/chembl_37_chemreps.txt.gz size MB: 292.5


## 1. Standardize structures consistently

For this initial 2D-graph experiment we retain the **largest molecular fragment** (discard salts), apply RDKit sanitization, and canonicalize the isomeric SMILES. We keep stereochemistry where explicitly encoded. This is a pragmatic first cleaning policy, not full medicinal-chemistry standardization; we do not claim it handles every tautomer, protonation state, or metal complex.

The **same** function is applied to ChEMBL and PPI molecules for overlap checking.

In [11]:
chooser = rdMolStandardize.LargestFragmentChooser(preferOrganic=True)
ALLOWED_ELEMENTS = {'H','B','C','N','O','F','P','S','Cl','Br','I'}

def clean_smiles(smiles):
    if not isinstance(smiles, str) or not smiles.strip():
        return None
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        mol = chooser.choose(mol)
        Chem.SanitizeMol(mol)
        n_heavy = mol.GetNumHeavyAtoms()
        if not (5 <= n_heavy <= 70):
            return None
        if any(a.GetSymbol() not in ALLOWED_ELEMENTS for a in mol.GetAtoms()):
            return None
        if not (60 <= Descriptors.MolWt(mol) <= 1000):
            return None
        return Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)
    except Exception:
        return None

ppi = pd.read_csv(PPI_CSV)
assert {'molecule_chembl_id', 'canonical_smiles'}.issubset(ppi.columns)
ppi_clean = {s for s in (clean_smiles(x) for x in ppi['canonical_smiles']) if s}
ppi_ids = set(ppi['molecule_chembl_id'].dropna().astype(str))
print('PPI original molecules:', len(ppi))
print('PPI standardized structures:', len(ppi_clean))
assert ppi_clean, 'No usable PPI structures — inspect input CSV'


PPI original molecules: 248
PPI standardized structures: 248


[11:08:23] Running LargestFragmentChooser
[11:08:23] Running LargestFragmentChooser
[11:08:23] Running LargestFragmentChooser
[11:08:23] Running LargestFragmentChooser
[11:08:23] Running LargestFragmentChooser
[11:08:23] Running LargestFragmentChooser
[11:08:23] Running LargestFragmentChooser
[11:08:23] Running LargestFragmentChooser
[11:08:23] Running LargestFragmentChooser
[11:08:23] Running LargestFragmentChooser
[11:08:23] Running LargestFragmentChooser
[11:08:23] Running LargestFragmentChooser
[11:08:23] Fragment: Br
[11:08:23] New largest fragment: Br (2)
[11:08:23] Fragment: N=C(N)Nc1nc2c(s1)CCc1ccccc1-2
[11:08:23] New largest fragment: N=C(N)Nc1nc2c(s1)CCc1ccccc1-2 (29)
[11:08:23] Running LargestFragmentChooser
[11:08:23] Fragment: Br
[11:08:23] New largest fragment: Br (2)
[11:08:23] Fragment: Cc1ccccc1-c1csc(NC(=N)NCc2ccccc2)n1
[11:08:23] New largest fragment: Cc1ccccc1-c1csc(NC(=N)NCc2ccccc2)n1 (41)
[11:08:23] Running LargestFragmentChooser
[11:08:23] Fragment: Br
[11:08:23]

## 2. Uniformly sample entries from the full ChEMBL file

Reading the complete compressed TSV into pandas is unnecessary. Reservoir sampling gives each ChEMBL **row** the same chance to enter our provisional subset, while using bounded memory. We intentionally draw more than `N_SAMPLE` entries to allow for filtering/deduplication.

**Important:** This is uniform *row* sampling, not uniform sampling of distinct chemical scaffolds. Since compounds may share chemistry, the final training distribution still reflects ChEMBL composition.

In [12]:
def reservoir_chemreps(path, k, seed=42):
    rng = random.Random(seed)
    reservoir = []
    n_seen = 0
    with gzip.open(path, 'rt', encoding='utf-8', errors='replace') as fh:
        header = fh.readline().rstrip('\n').split('\t')
        names = {c.strip().lower(): i for i, c in enumerate(header)}
        assert 'chembl_id' in names and 'canonical_smiles' in names, f'Unexpected header: {header}'
        i_id, i_smiles = names['chembl_id'], names['canonical_smiles']
        for line in fh:
            fields = line.rstrip('\n').split('\t')
            if len(fields) <= max(i_id, i_smiles):
                continue
            compound_id, smiles = fields[i_id], fields[i_smiles]
            if not smiles:
                continue
            n_seen += 1
            row = (compound_id, smiles)
            if len(reservoir) < k:
                reservoir.append(row)
            else:
                j = rng.randrange(n_seen)
                if j < k:
                    reservoir[j] = row
    return reservoir, n_seen

k = int(np.ceil(N_SAMPLE * RESERVOIR_MULTIPLIER))
raw_rows, eligible_rows = reservoir_chemreps(CHEMREPS, k, seed=SEED)
print('Eligible ChEMBL rows scanned:', eligible_rows)
print('Raw rows sampled:', len(raw_rows))

Eligible ChEMBL rows scanned: 2897819
Raw rows sampled: 150000


## 3. Filter structures, remove PPI molecules and deduplicate

Exclude known PPI molecules **by ID and by standardized SMILES**. Since a PPI molecule might be present under a different ChEMBL ID, the SMILES comparison matters. We keep one record per standardized structure.

The exclusion avoids **exact-compound overlap** with the later labeled PPI fine-tuning and testing dataset. It does *not* guarantee exclusion of all PPI-like scaffolds; we report scaffold overlap separately.

In [13]:
stats = Counter()
unique = {}
for chembl_id, smiles in raw_rows:
    stats['sampled'] += 1
    if chembl_id in ppi_ids:
        stats['ppi_id_removed'] += 1
        continue
    standard = clean_smiles(smiles)
    if standard is None:
        stats['invalid_or_filtered'] += 1
        continue
    if standard in ppi_clean:
        stats['ppi_structure_removed'] += 1
        continue
    if standard in unique:
        stats['duplicate_removed'] += 1
        continue
    unique[standard] = chembl_id

# Stable order; shuffle reproducibly before selecting final N_SAMPLE
rows = [(chembl_id, smiles) for smiles, chembl_id in unique.items()]
random.Random(SEED).shuffle(rows)
if len(rows) < N_SAMPLE:
    print(f'WARNING: only {len(rows)} clean unique molecules; increase RESERVOIR_MULTIPLIER or reduce N_SAMPLE')
rows = rows[:N_SAMPLE]
prepared = pd.DataFrame(rows, columns=['chembl_id', 'canonical_smiles'])
prepared['n_heavy_atoms'] = prepared['canonical_smiles'].map(
    lambda s: Chem.MolFromSmiles(s).GetNumHeavyAtoms())
prepared['scaffold'] = prepared['canonical_smiles'].map(
    lambda s: MurckoScaffold.MurckoScaffoldSmiles(smiles=s, includeChirality=False))
print('Filtering counts:', dict(stats))
print('Final molecules:', len(prepared))
print('Unique scaffolds:', prepared.scaffold.nunique())
assert not set(prepared.canonical_smiles).intersection(ppi_clean)
assert not set(prepared.chembl_id).intersection(ppi_ids)
display(prepared.head())

[11:08:28] Running LargestFragmentChooser
[11:08:28] Running LargestFragmentChooser
[11:08:28] Running LargestFragmentChooser
[11:08:28] Running LargestFragmentChooser
[11:08:28] Running LargestFragmentChooser
[11:08:28] Running LargestFragmentChooser
[11:08:28] Running LargestFragmentChooser
[11:08:28] Running LargestFragmentChooser
[11:08:28] Running LargestFragmentChooser
[11:08:28] Running LargestFragmentChooser
[11:08:28] Running LargestFragmentChooser
[11:08:28] Running LargestFragmentChooser
[11:08:28] Running LargestFragmentChooser
[11:08:28] Running LargestFragmentChooser
[11:08:28] Running LargestFragmentChooser
[11:08:28] Running LargestFragmentChooser
[11:08:28] Running LargestFragmentChooser
[11:08:28] Running LargestFragmentChooser
[11:08:28] Running LargestFragmentChooser
[11:08:28] Running LargestFragmentChooser
[11:08:28] Running LargestFragmentChooser
[11:08:28] Running LargestFragmentChooser
[11:08:28] Running LargestFragmentChooser
[11:08:28] Running LargestFragment

Filtering counts: {'sampled': 150000, 'invalid_or_filtered': 3429, 'ppi_id_removed': 9, 'duplicate_removed': 273, 'ppi_structure_removed': 6}
Final molecules: 100000
Unique scaffolds: 70427


,chembl_id,canonical_smiles,n_heavy_atoms,scaffold
0,CHEMBL324493,CCOc1onc2c1C(=O)/C(=N\c1ccccc1)C=C2O,21,O=C1C(=Nc2ccccc2)C=Cc2nocc21
1,CHEMBL1484854,COc1cc(C(=O)Nc2ccc3c(c2)OCCO3)ccc1OC(F)F,25,O=C(Nc1ccc2c(c1)OCCO2)c1ccccc1
2,CHEMBL2357578,CNC(=O)C1Cc2c(-c3ccccc3Oc3ccccc3)ccnc2O1,26,c1ccc(Oc2ccccc2-c2ccnc3c2CCO3)cc1
3,CHEMBL5806139,COc1ncc(-c2cc(C(F)(F)F)c3c(N)ncnn23)cc1C(=O)N[...,40,O=C(NC1CCN(Cc2ccccn2)C1)c1cncc(-c2ccc3cncnn23)c1
4,CHEMBL5970264,N#Cc1c(F)cc(-c2c(C(=O)N3CCC3)nn3c2OCCC3)cc1F,25,O=C(c1nn2c(c1-c1ccccc1)OCCC2)N1CCC1


## 4. Train/validation split for *self-supervised pretraining*

Use random 90/10 split of the **pretraining structures**, with unique compounds assigned to one side only. This validation set monitors the masked-atom objective, **not PPI bioactivity**.

Later, the PPI project will retain its existing held-out random and scaffold test partitions. They must never be used for early stopping or hyperparameter tuning.

In [14]:
train_df, val_df = train_test_split(
    prepared, test_size=VAL_FRACTION, random_state=SEED, shuffle=True
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
assert set(train_df.canonical_smiles).isdisjoint(set(val_df.canonical_smiles))

train_path = OUT / 'chembl37_pretrain_train.csv'
val_path = OUT / 'chembl37_pretrain_val.csv'
full_path = OUT / 'chembl37_pretrain_all.csv'
train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path, index=False)
prepared.to_csv(full_path, index=False)

ppi_scaffolds = {
    MurckoScaffold.MurckoScaffoldSmiles(smiles=s, includeChirality=False)
    for s in ppi_clean
}
shared = set(prepared.scaffold) & ppi_scaffolds
print('Training:', len(train_df), 'Validation:', len(val_df))
print('Scaffolds shared with PPI (audit only):', len(shared))
print('Molecules on shared scaffolds:', int(prepared.scaffold.isin(shared).sum()))
print('Saved:', train_path, val_path, full_path)

Training: 90000 Validation: 10000
Scaffolds shared with PPI (audit only): 21
Molecules on shared scaffolds: 280
Saved: data/processed/chembl37_pretrain_train.csv data/processed/chembl37_pretrain_val.csv data/processed/chembl37_pretrain_all.csv


## 5. Save experiment manifest and inspect the result

We log release, random seed, filters, and file digest. For publication-level work, also record the raw file checksum, dataset license and any subsequent preprocessing changes.

In [15]:
def sha256(path, chunk_size=1024*1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as fh:
        for chunk in iter(lambda: fh.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

manifest = {
    'source': 'ChEMBL 37 chemreps TSV.gz',
    'download_url': 'https://ftp.ebi.ac.uk/pub/databases/chembl/ChEMBLdb/releases/chembl_37/chembl_37_chemreps.txt.gz',
    'source_file': str(CHEMREPS),
    'source_sha256': sha256(CHEMREPS),
    'ppi_source_file': str(PPI_CSV),
    'ppi_source_sha256': sha256(PPI_CSV),
    'rdkit_version': rdkit.__version__,
    'seed': SEED,
    'requested_sample': N_SAMPLE,
    'actual_molecules': len(prepared),
    'train_molecules': len(train_df),
    'val_molecules': len(val_df),
    'filters': {
        'fragment': 'largest fragment, prefer organic',
        'heavy_atoms': [5, 70],
        'molecular_weight': [60, 1000],
        'elements': sorted(ALLOWED_ELEMENTS),
        'remove_exact_ppi_ids_and_standardized_structures': True,
        'remove_duplicate_standardized_smiles': True
    },
    'ppi_scaffolds_shared': len(shared),
    'sample_count_after_each_filter': dict(stats),
}
manifest_path = OUT / 'chembl37_pretrain_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Saved manifest:', manifest_path)
print('Processed files:', [p.name for p in sorted(OUT.glob('chembl37_pretrain_*'))])
print('Ready for notebook 02: masked-atom GINE pretraining')

Saved manifest: data/processed/chembl37_pretrain_manifest.json
Processed files: ['chembl37_pretrain_all.csv', 'chembl37_pretrain_manifest.json', 'chembl37_pretrain_train.csv', 'chembl37_pretrain_val.csv']
Ready for notebook 02: masked-atom GINE pretraining


## What this notebook establishes — and what it does not

- **Accomplished:** sampled a release-pinned ChEMBL structure corpus; applied consistent structural cleaning; removed exact PPI overlaps; created training/validation datasets and manifest.
- **Not accomplished:** GINE weights, masked-atom training, molecular activity labels, PPI prediction or a generative Transformer.
- **Next notebook:** `02_GINE_Pretraining.ipynb` will encode atom/bond features, mask a fraction of atoms, train the encoder to predict the masked atom identities, validate the self-supervised objective, and save reusable pretrained weights.
- **Scientific limitation:** the pretraining and downstream collections can still share similar molecular scaffolds, and no chemistry representation objective guarantees better pIC₅₀ prediction. This must be tested against our Random Forest and scratch-GINE baselines.